# Notebook 05 - Drug-likeness Filter Pipeline

**Inputs:**
- `results/generated/1M17_raw.sdf` (998 unique mols)
- `results/generated/4I22_raw.sdf` (989 unique mols)

**Outputs (in `results/filtered/`):**
- `1M17_all_metrics.csv`, `4I22_all_metrics.csv` - full per-molecule metrics for every input
- `1M17_filtered.csv`,    `4I22_filtered.csv`    - only molecules passing all hard filters
- `baselines.csv` - same metrics computed for Erlotinib / Gefitinib / Osimertinib (reference)

**Pipeline (each row = one rule, applied independently then combined):**

| Stage | Rule | Why |
|-------|------|-----|
| Drug-like | QED >= 0.4 | weighted desirability score |
| Synthesizable | SA <= 5 | Ertl & Schuffenhauer score |
| Oral bioavailability | <=1 Lipinski Ro5 violation | classical rule |
| Toxicophore safety | 0 PAINS alerts | flags pan-assay interference |
| Reasonable size | 200 <= MW <= 600 | drug-like MW range |
| Reasonable lipophilicity | -2 <= LogP <= 6 | absorption window |

**Runtime:** CPU-only, ~1 minute total.


In [ ]:
# -- Cell 1: Bootstrap (CPU runtime is fine; no GPU / no conda needed) --
import os, sys, subprocess

# Mount Drive (idempotent)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
except ImportError:
    # Running locally: assume cwd is repo root
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

# Pull latest code (in case src/filtering.py was added/updated since last session)
subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

# Make sure rdkit is available (cheap if already installed)
try:
    import rdkit
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rdkit'], check=True)
    import rdkit
print('rdkit:', rdkit.__version__)


## Step 1 - Load data: 2 generated SDFs + 3 baseline drugs

In [ ]:
# -- Cell 2: Load SDFs and prepare baseline reference --
from rdkit import Chem
from rdkit.Chem import AllChem
from src.utils import load_config
from src.filtering import load_sdf_mols, morgan_fp

# Load target/baseline definitions
cfg = load_config('configs/targets.yaml')
baseline_smis = {k: v['smiles'] for k, v in cfg['baselines'].items()}
baseline_names = list(baseline_smis.keys())
baseline_mols  = [Chem.MolFromSmiles(s) for s in baseline_smis.values()]
baseline_fps   = [morgan_fp(m) for m in baseline_mols]

print('Baselines loaded:')
for name, smi in baseline_smis.items():
    print(f'  {name:14s} {smi}')

# Load generated molecules
wt_mols  = load_sdf_mols(f'{PROJECT_ROOT}/results/generated/1M17_raw.sdf')
mut_mols = load_sdf_mols(f'{PROJECT_ROOT}/results/generated/4I22_raw.sdf')
print(f'\n1M17 (WT):    {len(wt_mols)} parseable molecules')
print(f'4I22 (T790M): {len(mut_mols)} parseable molecules')


## Step 2 - Compute all metrics

For every molecule we compute MW, LogP, HBA, HBD, TPSA, RotBonds, QED, SA score,
Lipinski violations, PAINS alerts, and the max Tanimoto similarity to the
three approved baselines (so we can later spot mols that "look like" Erlotinib etc.).


In [ ]:
# -- Cell 3: Compute per-molecule metrics --
from src.filtering import compute_all_metrics

wt_df  = compute_all_metrics(wt_mols,  baseline_fps, baseline_names)
mut_df = compute_all_metrics(mut_mols, baseline_fps, baseline_names)

# Also for the baselines themselves (so the report has the same columns)
baseline_df = compute_all_metrics(baseline_mols, baseline_fps, baseline_names)
baseline_df['name'] = baseline_names

print(f'WT metrics  : {len(wt_df)} rows, {len(wt_df.columns)} columns')
print(f'Mut metrics : {len(mut_df)} rows')
print(f'Baselines   : {len(baseline_df)} rows')

print('\nFirst few WT rows:')
wt_df.head(3)


## Step 3 - Apply filters and report attrition

Each rule is applied independently first (to see where attrition happens),
then combined into the final filter mask.


In [ ]:
# -- Cell 4: Apply hard filters and print the funnel --
from src.filtering import apply_filters

wt_filt,  wt_summary  = apply_filters(wt_df)
mut_filt, mut_summary = apply_filters(mut_df)

def print_funnel(title, summary):
    n0 = summary['input']
    print(f'\n=== {title} (start: {n0}) ===')
    for k, v in summary.items():
        if k == 'input': continue
        pct = 100 * v / n0
        bar = '#' * int(pct/2)
        print(f'  {k:42s} {v:4d}  ({pct:5.1f}%)  {bar}')

print_funnel('1M17 (WT)',  wt_summary)
print_funnel('4I22 (T790M)', mut_summary)

print(f'\nFinal: {len(wt_filt)} WT mols, {len(mut_filt)} mutant mols passed all filters.')


## Step 4 - Quick sanity check on baselines

Sanity: Erlotinib / Gefitinib / Osimertinib should themselves PASS all the filters
(they are approved drugs). If any of them fails, our filter thresholds are too strict.


In [ ]:
# -- Cell 5: Verify baselines pass the filter --
baseline_filt, baseline_summary = apply_filters(baseline_df)

print('Baseline molecules - per-rule status:')
for _, row in baseline_df.iterrows():
    print(f"  {row['name']:14s} QED={row['QED']:.2f}  SA={row['SA']:.2f}  "
          f"LipV={row['LipinskiViolations']}  PAINS={row['PAINS_Alerts']}  "
          f"MW={row['MW']:.0f}  LogP={row['LogP']:.2f}")

print(f'\n{len(baseline_filt)}/{len(baseline_df)} baselines pass all filters.')
if len(baseline_filt) < len(baseline_df):
    print('WARNING: some baselines fail. Filter thresholds may be too strict.')


## Step 5 - Save outputs

In [ ]:
# -- Cell 6: Save all four CSVs --
import os
out_dir = f'{PROJECT_ROOT}/results/filtered'
os.makedirs(out_dir, exist_ok=True)

wt_df.to_csv(f'{out_dir}/1M17_all_metrics.csv', index=False)
mut_df.to_csv(f'{out_dir}/4I22_all_metrics.csv', index=False)
wt_filt.to_csv(f'{out_dir}/1M17_filtered.csv',  index=False)
mut_filt.to_csv(f'{out_dir}/4I22_filtered.csv', index=False)
baseline_df.to_csv(f'{out_dir}/baselines.csv',  index=False)

print('Saved:')
for f in sorted(os.listdir(out_dir)):
    p = os.path.join(out_dir, f)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')


## Step 6 - Visual sanity check

Show the **top 6 WT molecules by QED** that survived filtering. Useful to
eyeball that they look chemically reasonable.


In [ ]:
# -- Cell 7: Draw top-6 by QED for WT and Mutant --
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

def draw_top(df, n=6, title='', sort_col='QED'):
    top = df.sort_values(sort_col, ascending=False).head(n)
    mols = [Chem.MolFromSmiles(s) for s in top['smiles']]
    legends = [f"QED={q:.2f} SA={s:.1f}\nMW={m:.0f} LogP={l:.1f}"
               for q, s, m, l in zip(top['QED'], top['SA'], top['MW'], top['LogP'])]
    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(280, 220),
                                legends=legends)
    print(f'--- {title} (top {n} by {sort_col}) ---')
    display(img)

draw_top(wt_filt,  6, 'WT (1M17)')
draw_top(mut_filt, 6, 'Mutant (4I22)')


## Step 7 - Quick distribution comparison

Look at the QED and SA score distributions: do filtered WT and Mutant molecules
sit in similar property ranges? If yes, downstream comparisons are apples-to-apples.


In [ ]:
# -- Cell 8: QED + SA distributions side by side --
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for df, label, color in [(wt_filt, 'WT (1M17)', '#1f77b4'),
                          (mut_filt, 'Mutant (4I22)', '#d62728')]:
    axes[0].hist(df['QED'], bins=20, alpha=0.55, label=label, color=color)
    axes[1].hist(df['SA'],  bins=20, alpha=0.55, label=label, color=color)

# Mark baseline drug positions as vertical lines
for _, row in baseline_df.iterrows():
    axes[0].axvline(row['QED'], linestyle=':', linewidth=1)
    axes[1].axvline(row['SA'],  linestyle=':', linewidth=1)
    axes[0].text(row['QED'], axes[0].get_ylim()[1]*0.95, row['name'][:3],
                 ha='center', va='top', fontsize=8)

axes[0].set_xlabel('QED'); axes[0].set_ylabel('count'); axes[0].set_title('QED distribution')
axes[1].set_xlabel('SA score'); axes[1].set_title('SA distribution')
axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.show()


## What's next

After this notebook, you have:

- **2 ranked, drug-like, PAINS-free shortlists** (CSV) ready for docking
- A clear funnel showing how 998/989 raw mols collapse to ~150-300 drug-like ones
- The same metrics computed on Erlotinib / Gefitinib / Osimertinib for direct comparison

**Next: Notebook 07** runs AutoDock Vina cross-docking on these shortlists.
Each filtered molecule gets scored against BOTH the WT pocket and the T790M pocket,
producing the (vina_WT, vina_Mut) pairs that feed Notebook 09's selectivity scatter.
